# Discovery Agent — Google Places API
**IST 488/688 Final Project — Milestone (Apr 19, 2026)**
**Owner:** Lauren

> ⚠️ **Backup draft** prepared by Ryan as a starting point. Replace mock data with live API calls when your `GOOGLE_PLACES_API_KEY` is ready, or use this as scaffolding.

**Pipeline position:**
`[THIS LAYER: Discovery Agent] → Ryan's scraper → Leytisha's enrichment → Toby's ChromaDB`

**What this notebook does:**
1. Calls the **new Places API (Text Search)** to pull restaurants for Syracuse, Rochester, and Albany.
2. Normalizes results into a consistent metadata schema (one shape per restaurant, no matter the city).
3. Falls back to bundled mock data if no API key is present — so the rest of the pipeline can develop in parallel.
4. Saves output to `restaurants_raw.json` for the downstream agents to consume.

**API used:** [Places API (New)](https://developers.google.com/maps/documentation/places/web-service/text-search) — POST endpoint with field masks, NOT the legacy v3.


## 1. Install dependencies

In [ ]:
!pip install -q requests


## 2. Imports & config

In [ ]:
import os
import json
import time
from pathlib import Path
from typing import Optional

import requests

# Set this in Colab via:  os.environ["GOOGLE_PLACES_API_KEY"] = "..."
# Or use Colab Secrets:    from google.colab import userdata; key = userdata.get("GOOGLE_PLACES_API_KEY")
API_KEY = os.environ.get("GOOGLE_PLACES_API_KEY", "")

PLACES_TEXT_SEARCH_URL = "https://places.googleapis.com/v1/places:searchText"

# Field mask — REQUIRED by the new API. Only request what we need (saves cost; new API bills by field tier).
FIELD_MASK = ",".join([
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.rating",
    "places.userRatingCount",
    "places.priceLevel",
    "places.websiteUri",
    "places.types",
])

TARGET_CITIES = ["Syracuse, NY", "Rochester, NY", "Albany, NY"]
RESULTS_PER_CITY = 7   # 7 × 3 = 21 ≈ 20-restaurant milestone target


## 3. Bundled mock data (fallback when no API key is set)

In [ ]:
SAMPLE_RESTAURANTS = [
    # --- Syracuse ---
    {"place_id": "syr_001", "name": "Dinosaur Bar-B-Que", "address": "246 W Willow St, Syracuse, NY 13202",
     "city": "Syracuse", "rating": 4.5, "user_rating_count": 6800, "price_level": "MODERATE",
     "website_url": "https://www.dinosaurbarbque.com/", "types": ["restaurant", "barbecue_restaurant"]},
    {"place_id": "syr_002", "name": "Pastabilities", "address": "311 S Franklin St, Syracuse, NY 13202",
     "city": "Syracuse", "rating": 4.4, "user_rating_count": 2100, "price_level": "MODERATE",
     "website_url": "https://pastabilities.com/", "types": ["restaurant", "italian_restaurant"]},
    {"place_id": "syr_003", "name": "Apizza Regionale", "address": "260 W Genesee St, Syracuse, NY 13202",
     "city": "Syracuse", "rating": 4.4, "user_rating_count": 950, "price_level": "MODERATE",
     "website_url": "https://apizzaregionale.com/", "types": ["restaurant", "pizza_restaurant"]},
    {"place_id": "syr_004", "name": "Phoebe's Restaurant", "address": "900 E Genesee St, Syracuse, NY 13210",
     "city": "Syracuse", "rating": 4.3, "user_rating_count": 780, "price_level": "MODERATE",
     "website_url": "https://phoebesrestaurant.com/", "types": ["restaurant", "american_restaurant"]},
    {"place_id": "syr_005", "name": "Kitty Hoynes Irish Pub", "address": "301 W Fayette St, Syracuse, NY 13202",
     "city": "Syracuse", "rating": 4.4, "user_rating_count": 1500, "price_level": "MODERATE",
     "website_url": "https://kittyhoynes.com/", "types": ["restaurant", "irish_pub"]},
    {"place_id": "syr_006", "name": "Otro Cinco", "address": "206 S Warren St, Syracuse, NY 13202",
     "city": "Syracuse", "rating": 4.3, "user_rating_count": 620, "price_level": "MODERATE",
     "website_url": "https://otrocincorestaurant.com/", "types": ["restaurant", "mexican_restaurant"]},
    {"place_id": "syr_007", "name": "The Fish Friar", "address": "247 W Fayette St, Syracuse, NY 13202",
     "city": "Syracuse", "rating": 4.5, "user_rating_count": 540, "price_level": "EXPENSIVE",
     "website_url": "https://www.fishfriar.com/", "types": ["restaurant", "seafood_restaurant"]},
    # --- Rochester ---
    {"place_id": "roc_001", "name": "Nick Tahou Hots", "address": "320 W Main St, Rochester, NY 14608",
     "city": "Rochester", "rating": 4.2, "user_rating_count": 2300, "price_level": "INEXPENSIVE",
     "website_url": "https://nicktahous.com/", "types": ["restaurant", "american_restaurant"]},
    {"place_id": "roc_002", "name": "Dinosaur Bar-B-Que Rochester", "address": "99 Court St, Rochester, NY 14604",
     "city": "Rochester", "rating": 4.5, "user_rating_count": 4900, "price_level": "MODERATE",
     "website_url": "https://www.dinosaurbarbque.com/", "types": ["restaurant", "barbecue_restaurant"]},
    {"place_id": "roc_003", "name": "Good Luck", "address": "50 Anderson Ave, Rochester, NY 14607",
     "city": "Rochester", "rating": 4.6, "user_rating_count": 1100, "price_level": "EXPENSIVE",
     "website_url": "https://restaurantgoodluck.com/", "types": ["restaurant", "american_restaurant"]},
    {"place_id": "roc_004", "name": "Han Noodle Bar", "address": "687 Monroe Ave, Rochester, NY 14607",
     "city": "Rochester", "rating": 4.5, "user_rating_count": 1800, "price_level": "MODERATE",
     "website_url": "https://hannoodlebar.com/", "types": ["restaurant", "chinese_restaurant"]},
    {"place_id": "roc_005", "name": "Lento", "address": "274 N Goodman St, Rochester, NY 14607",
     "city": "Rochester", "rating": 4.5, "user_rating_count": 700, "price_level": "EXPENSIVE",
     "website_url": "https://lentorestaurant.com/", "types": ["restaurant", "american_restaurant"]},
    {"place_id": "roc_006", "name": "Atlas Eats", "address": "2185 N Clinton Ave, Rochester, NY 14621",
     "city": "Rochester", "rating": 4.6, "user_rating_count": 850, "price_level": "MODERATE",
     "website_url": "https://atlaseats.com/", "types": ["restaurant", "mediterranean_restaurant"]},
    # --- Albany ---
    {"place_id": "alb_001", "name": "Cafe Capriccio", "address": "49 Grand St, Albany, NY 12202",
     "city": "Albany", "rating": 4.6, "user_rating_count": 920, "price_level": "EXPENSIVE",
     "website_url": "https://cafecapriccio.com/", "types": ["restaurant", "italian_restaurant"]},
    {"place_id": "alb_002", "name": "Jack's Oyster House", "address": "42 State St, Albany, NY 12207",
     "city": "Albany", "rating": 4.4, "user_rating_count": 1300, "price_level": "EXPENSIVE",
     "website_url": "https://jacksoysterhouse.com/", "types": ["restaurant", "seafood_restaurant"]},
    {"place_id": "alb_003", "name": "The Hollow Bar + Kitchen", "address": "79 N Pearl St, Albany, NY 12207",
     "city": "Albany", "rating": 4.4, "user_rating_count": 1100, "price_level": "MODERATE",
     "website_url": "https://thehollowalbany.com/", "types": ["restaurant", "american_restaurant"]},
    {"place_id": "alb_004", "name": "El Loco Mexican Cafe", "address": "465 Madison Ave, Albany, NY 12208",
     "city": "Albany", "rating": 4.4, "user_rating_count": 950, "price_level": "MODERATE",
     "website_url": "https://ellococafe.com/", "types": ["restaurant", "mexican_restaurant"]},
    {"place_id": "alb_005", "name": "Yono's Restaurant", "address": "25 Chapel St, Albany, NY 12210",
     "city": "Albany", "rating": 4.6, "user_rating_count": 580, "price_level": "EXPENSIVE",
     "website_url": "https://yonosrestaurant.com/", "types": ["restaurant", "fine_dining"]},
    {"place_id": "alb_006", "name": "Cardona's Market", "address": "340 Delaware Ave, Albany, NY 12209",
     "city": "Albany", "rating": 4.6, "user_rating_count": 720, "price_level": "MODERATE",
     "website_url": "https://cardonasmarket.com/", "types": ["restaurant", "italian_restaurant"]},
    {"place_id": "alb_007", "name": "New World Bistro Bar", "address": "300 Delaware Ave, Albany, NY 12209",
     "city": "Albany", "rating": 4.4, "user_rating_count": 880, "price_level": "MODERATE",
     "website_url": "https://newworldbistrobar.com/", "types": ["restaurant", "american_restaurant"]},
]


## 4. Live API call (when key is available)

In [ ]:
def search_restaurants_live(city: str, max_results: int = 7) -> list[dict]:
    """Call the new Places API (Text Search) for restaurants in a city."""
    if not API_KEY:
        raise RuntimeError("GOOGLE_PLACES_API_KEY not set")

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": FIELD_MASK,
    }
    payload = {
        "textQuery": f"restaurants in {city}",
        "maxResultCount": max_results,
        "includedType": "restaurant",
    }
    r = requests.post(PLACES_TEXT_SEARCH_URL, headers=headers, json=payload, timeout=20)
    r.raise_for_status()
    return r.json().get("places", [])


def normalize_place(raw: dict, city: str) -> dict:
    """Convert a raw Places API response into our pipeline's standard schema."""
    return {
        "place_id": raw.get("id", ""),
        "name": (raw.get("displayName") or {}).get("text", ""),
        "address": raw.get("formattedAddress", ""),
        "city": city.split(",")[0].strip(),
        "rating": raw.get("rating"),
        "user_rating_count": raw.get("userRatingCount"),
        "price_level": raw.get("priceLevel"),  # e.g. "PRICE_LEVEL_MODERATE"
        "website_url": raw.get("websiteUri"),
        "types": raw.get("types", []),
    }


## 5. Discovery agent — main entry point
Returns a flat list of restaurant dicts. Uses live API when available, mock data otherwise.

In [ ]:
def discover_restaurants(cities: list[str] = None, per_city: int = RESULTS_PER_CITY) -> list[dict]:
    cities = cities or TARGET_CITIES
    out = []

    if not API_KEY:
        print("⚠️  No GOOGLE_PLACES_API_KEY set — returning bundled mock data.")
        wanted_cities = {c.split(",")[0].strip() for c in cities}
        return [r for r in SAMPLE_RESTAURANTS if r["city"] in wanted_cities]

    for city in cities:
        print(f"Searching: {city}")
        try:
            raw_places = search_restaurants_live(city, max_results=per_city)
            normalized = [normalize_place(p, city) for p in raw_places]
            # Filter out anything missing a website URL (Ryan's scraper needs it)
            normalized = [r for r in normalized if r.get("website_url")]
            print(f"  → {len(normalized)} restaurants with websites")
            out.extend(normalized)
            time.sleep(0.5)  # gentle rate limiting
        except requests.HTTPError as e:
            print(f"  ✗ HTTP error for {city}: {e}")
        except Exception as e:
            print(f"  ✗ Unexpected error for {city}: {e}")

    return out


## 6. Run discovery + save

In [ ]:
restaurants = discover_restaurants()
print(f"\nTotal restaurants discovered: {len(restaurants)}")

out_path = Path("restaurants_raw.json")
out_path.write_text(json.dumps(restaurants, indent=2))
print(f"Saved to: {out_path.resolve()}")

# Preview
for r in restaurants[:3]:
    print(f"  • {r['name']} ({r['city']}) — {r.get('rating')}★ — {r.get('website_url')}")


## 7. Milestone checklist (Apr 19)

- [x] Google Places API call structure (new API, with field mask)
- [x] Returns Syracuse, Rochester, Albany restaurants
- [x] Normalized output schema for downstream agents
- [x] Mock data fallback so teammates aren't blocked on your API key
- [x] Saves to `restaurants_raw.json`

## Apr 26 follow-ups (your scope)
- **Caching layer** — store responses keyed by `(city, query)` to avoid re-billing on dev iterations.
- **Cost tracking** — log estimated cost per call. Text Search w/ Essentials field mask runs ~$0.005–$0.017 per call depending on field tier.
- **Expand to remaining target NY cities** (Buffalo, Binghamton, Ithaca, etc. — coordinate with team).
- **Pagination** — for cities with >20 strong results, use `pageToken` to grab a second page.
